# Lab 21-10 · Antennas, Arrays, Beamforming and mmWave

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import antenna as ant


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · A patch and the Chu limit

*Predict* how the patch length changes from 28 to 39 GHz, and how the Chu bandwidth of a Bluetooth antenna changes when its sphere shrinks from 10 to 5 mm.

In [2]:
for f, er, h in ((28e9, 3.0, 0.25e-3), (39e9, 3.5, 0.2e-3)):
    w, e, dl, length = ant.patch_design(f, er, h)
    print(f"{f/1e9:.0f} GHz: W {w*1e3:.3f} mm, eps_eff {e:.3f}, dL {dl*1e3:.3f} mm, L {length*1e3:.3f} mm")
lam = ant.C0 / 2.44175e9
for r in (0.010, 0.0075, 0.005):
    ka = 2 * math.pi * r / lam
    q = ant.chu_q(ka)
    print(f"radius {r*1e3:.1f} mm: ka {ka:.3f}, Q >= {q:.1f}, VSWR-2 bandwidth <= {ant.bandwidth_from_q(q)*100:.2f} %")
ka = np.geomspace(0.1, 1.5, 300)
plt.figure(); plt.loglog(ka, ant.chu_q(ka)); plt.xlabel("ka"); plt.ylabel("minimum Q"); plt.show()

28 GHz: W 3.785 mm, eps_eff 2.747, dL 0.122 mm, L 2.986 mm
39 GHz: W 2.562 mm, eps_eff 3.148, dL 0.094 mm, L 1.977 mm
radius 10.0 mm: ka 0.512, Q >= 9.4, VSWR-2 bandwidth <= 7.51 %
radius 7.5 mm: ka 0.384, Q >= 20.3, VSWR-2 bandwidth <= 3.48 %
radius 5.0 mm: ka 0.256, Q >= 63.6, VSWR-2 bandwidth <= 1.11 %


## Part 2 · Arrays: scan loss and grating lobes

*Predict* where the grating lobe of a 16-element array with 0.8λ spacing appears when steered to 30°.

In [3]:
th = np.linspace(-90, 90, 36001)
af = ant.array_factor(th, 16, 0.8, 30)
neg = np.where(th < 0, af, 0)
print(f"grating lobe at {th[np.argmax(neg)]:.1f} deg, level {20*np.log10(neg.max()):.2f} dB")
print(f"max scan without grating lobes: 0.5 lambda {ant.max_scan_no_grating_deg(0.5):.1f} deg, 0.8 lambda {ant.max_scan_no_grating_deg(0.8):.1f} deg")
plt.figure()
for d, lab_ in ((0.5, "0.5 lambda"), (0.8, "0.8 lambda")):
    plt.plot(th, 20 * np.log10(np.maximum(ant.array_factor(th, 16, d, 30), 1e-4)), label=lab_)
plt.ylim(-40, 2); plt.xlabel("angle (deg)"); plt.ylabel("|AF| (dB)"); plt.legend(); plt.show()
for t0 in (0, 30, 45, 60):
    print(f"scan {t0:2d} deg: element (cos) loss {-10*math.log10(math.cos(math.radians(t0))):.2f} dB, "
          f"16-element beamwidth {ant.hpbw_deg(16, 0.5, t0):.2f} deg")

grating lobe at -48.6 deg, level -0.00 dB
max scan without grating lobes: 0.5 lambda 90.0 deg, 0.8 lambda 14.5 deg


scan  0 deg: element (cos) loss -0.00 dB, 16-element beamwidth 6.36 deg
scan 30 deg: element (cos) loss 0.62 dB, 16-element beamwidth 7.35 deg


scan 45 deg: element (cos) loss 1.51 dB, 16-element beamwidth 9.02 deg
scan 60 deg: element (cos) loss 3.01 dB, 16-element beamwidth 12.99 deg


## Part 3 · Beam squint

*Predict* the beam direction at 26 and 30 GHz for a 16-element array steered to 60° at 28 GHz, and the gain loss toward 60°.

In [4]:
for f in (26e9, 28e9, 30e9):
    fr = f / 28e9
    print(f"{f/1e9:.0f} GHz: beam at {ant.squint_angle_deg(fr, 60):.2f} deg, toward 60 deg "
          f"{20*math.log10(ant.array_factor([60], 16, 0.5, 60, fr)[0]):.2f} dB (TTD "
          f"{20*math.log10(ant.array_factor([60], 16, 0.5, 60, fr, 'ttd')[0]):.2f} dB)")
print(f"3 dB squint bandwidth {ant.squint_bandwidth(16, 0.5, 60)*28:.2f} GHz (large-N formula {ant.squint_bandwidth(16, 0.5, 60, exact=False)*28:.2f} GHz)")
s = ngspice("squint.cir")
print({k: round(s[k], 3) for k in ("lps26", "lps28", "lps30", "gttd26")})
th = np.linspace(30, 90, 2001)
plt.figure()
for f in (26e9, 28e9, 30e9):
    plt.plot(th, 20 * np.log10(np.maximum(ant.array_factor(th, 16, 0.5, 60, f / 28e9), 1e-4)), label=f"{f/1e9:.0f} GHz")
plt.ylim(-30, 2); plt.xlabel("angle (deg)"); plt.ylabel("|AF| (dB)"); plt.legend(); plt.show()

26 GHz: beam at 68.85 deg, toward 60 deg -3.82 dB (TTD 0.00 dB)
28 GHz: beam at 60.00 deg, toward 60 deg 0.00 dB (TTD 0.00 dB)
30 GHz: beam at 53.93 deg, toward 60 deg -3.82 dB (TTD 0.00 dB)


3 dB squint bandwidth 3.59 GHz (large-N formula 3.58 GHz)


{'lps26': -3.82, 'lps28': 0.0, 'lps30': -3.82, 'gttd26': 24.082}


## Part 4 · Phase quantization

*Predict* the gain loss and the floor at the first null for 3-bit phase shifters on 64 elements.

In [5]:
for b in (3, 4):
    peak, null = ant.quantization_monte_carlo(64, b)
    x = math.pi / 2 ** b
    S = (math.sin(x) / x) ** 2
    print(f"{b} bits: Monte Carlo loss {-10*math.log10(peak):.3f} dB, floor {10*math.log10(null):.2f} dB; "
          f"formula loss {-10*math.log10(S + (1 - S)/64):.3f} dB, floor {10*math.log10((1 - S)/64):.2f} dB")

3 bits: Monte Carlo loss 0.221 dB, floor -31.04 dB; formula loss 0.221 dB, floor -31.04 dB
4 bits: Monte Carlo loss 0.055 dB, floor -37.04 dB; formula loss 0.055 dB, floor -36.99 dB


## Part 5 · Element count and architecture power

*Predict* the element count that minimizes DC power for 60 dBm EIRP (5 dBi elements, 15 % PAs, 100 mW overhead).

In [6]:
E, ge, eta, poh = 1000.0, 10 ** 0.5, 0.15, 0.1
nopt = ant.optimal_elements(E, ge, eta, poh)
print(f"N* = {nopt:.1f}, DC power {ant.dc_power_w(nopt, E, ge, eta, poh):.2f} W")
for n in (32, 64, 128, 256, 512):
    print(f"N = {n:3d}: {ant.dc_power_w(n, E, ge, eta, poh):6.2f} W, PA {10*math.log10(E/(n**2*ge)*1e3):5.1f} dBm")
p = ant.adc_power_w(20e-15, 8, 1e9)
chain = 2 * p + 0.040
print(f"ADC {p*1e3:.2f} mW; digital {64*chain:.2f} W, hybrid (4 chains) {4*chain + 64*0.015:.2f} W, analog {chain + 64*0.015:.2f} W")

N* = 145.2, DC power 29.04 W
N =  32:  69.08 W, PA  24.9 dBm
N =  64:  39.34 W, PA  18.9 dBm
N = 128:  29.27 W, PA  12.9 dBm
N = 256:  33.84 W, PA   6.8 dBm
N = 512:  55.32 W, PA   0.8 dBm
ADC 5.12 mW; digital 3.22 W, hybrid (4 chains) 1.16 W, analog 1.01 W


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.